## Install Libraries

Translates the Vietnamese gold JSONL (`vi_sentence_str`, `vi_term`) into
English with a local Qwen checkpoint, and writes a new JSONL into the same
existing Drive folder.

Term mentions are translated once each (a glossary), and every sentence is
translated with its mentions replaced by `[[T0]]`, `[[T1]]`, ... placeholders.
The placeholders are substituted back with the glossary translations, so

- `en_term` is the glossary entry of the mention,
- `en_sentence_str` contains that exact term at the exact place the model put
  the placeholder,
- `en_start_token_idx` / `en_end_token_idx` follow from the substitution, so
  they are computed rather than guessed by searching the sentence.

The placeholder path is verified per sentence: a sentence whose placeholders do
not survive intact is not cached, and its plain translation is stored instead
so the span can still be searched; a mention that survives neither route gets
`null` indices and is reported, so a half-aligned dataset cannot pass silently.

`cui`, `icd10` and `label` are copied unchanged: only the two Vietnamese text
fields get an English counterpart. The notebook needs no repository clone: it
reads and writes JSONL only. Translations are cached in `CACHE_PATH` and
appended as they arrive, so a disconnected runtime resumes instead of starting
over.

In [ ]:
%pip install "tqdm>=4.66" "transformers>=4.49" "accelerate>=0.34" "bitsandbytes>=0.43"


## Import Libraries

`torch` and `transformers` are imported here; the model itself is loaded in
the model stage so a configuration mistake fails before the weights download.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import sys
import time
import unicodedata
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Constants and Configuration

Set `DRIVE_DATA_DIR` to the folder that already holds the gold JSONL; the
notebook refuses to create it, so a typo cannot scatter files over Drive.

`OUTPUT_PATH` is the translated dataset, `CACHE_PATH` is the append-only
translation memory used to resume. Both live in that same folder.

`LIMIT = None` translates every record; set an int for a smoke run.


In [ ]:
# ---------------------------------------------------------------------------
# Google Drive input and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
SOURCE_DATA_PATH = DRIVE_DATA_DIR / "test_fixed_verified_gold_revised.jsonl"
OUTPUT_DIR = DRIVE_DATA_DIR                                     # must already exist
OUTPUT_PATH = OUTPUT_DIR / "test_fixed_verified_gold_revised_en.jsonl"
CACHE_PATH = OUTPUT_DIR / "vi_en_translation_cache.jsonl"

# ---------------------------------------------------------------------------
# Translation
# ---------------------------------------------------------------------------
SOURCE_LANGUAGE = "Vietnamese"
TARGET_LANGUAGE = "English"
BATCH_SIZE = 8                  # strings per generate() call
MAX_INPUT_TOKENS = 512          # prompt cap; a truncated prompt is a bug
MAX_NEW_TOKENS = 256            # one sentence or term, never long
LIMIT = None                    # None = all records; an int truncates

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen3.8-27B"   # Hub id or a local directory
LOAD_IN_4BIT = True             # 27B in bf16 is ~54 GB; 4-bit is ~15 GB
DEVICE_MAP = "auto"
TRUST_REMOTE_CODE = False

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Input data   : {SOURCE_DATA_PATH}")
print(f"Output       : {OUTPUT_PATH}")
print(f"Cache        : {CACHE_PATH}")
print(f"Direction    : {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"Model        : {MODEL_ID} (4-bit={LOAD_IN_4BIT})")


## Stage 1 — Mount Drive and Locate the Existing Folder

Both the input and the two output files live in the folder you configured;
nothing is created outside it. The folder must exist already.


In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

source_path = Path(require_configured(SOURCE_DATA_PATH, "SOURCE_DATA_PATH"))
output_dir = Path(require_configured(OUTPUT_DIR, "OUTPUT_DIR"))
output_path = Path(OUTPUT_PATH)
cache_path = Path(CACHE_PATH)

if not source_path.is_file():
    raise FileNotFoundError(f"Gold data not found: {source_path}")
if not output_dir.is_dir():
    raise FileNotFoundError(
        f"OUTPUT_DIR does not exist: {output_dir}. Point it at the existing folder; "
        "this notebook never creates it."
    )
for label, path in (("OUTPUT_PATH", output_path), ("CACHE_PATH", cache_path)):
    if path.parent != output_dir:
        raise ValueError(f"{label} must live in OUTPUT_DIR, got {path.parent}")
log(f"Input: {source_path} ({source_path.stat().st_size / 1024 ** 2:.2f} MB)")
log(f"Output dir: {output_dir}")


## Stage 2 — Load the Gold Records

Every non-blank line must be an object with `entities` and
`vi_sentence_str`; a malformed record fails here instead of producing a
half-translated file.


In [ ]:
log(f"Loading gold records from {source_path} ...")
records = []
with source_path.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, 1):
        if not line.strip():
            continue
        where = f"{source_path}:{line_number}"
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{where}: invalid JSON: {exc.msg}") from exc
        if not isinstance(record, dict):
            raise ValueError(f"{where}: record must be an object")
        sentence = record.get("vi_sentence_str")
        entities = record.get("entities")
        if not isinstance(sentence, str) or not sentence.strip():
            raise ValueError(f"{where}: vi_sentence_str must be a non-empty string")
        if not isinstance(entities, list) or not entities:
            raise ValueError(f"{where}: entities must be a non-empty list")
        for entity_index, entity in enumerate(entities):
            term = entity.get("vi_term") if isinstance(entity, dict) else None
            if not isinstance(term, str) or not term.strip():
                raise ValueError(f"{where}:entities[{entity_index}]: vi_term must be a non-empty string")
        records.append(record)
        if LIMIT is not None and len(records) >= LIMIT:
            break

if not records:
    raise ValueError(f"{source_path} produced no records.")
n_entities = sum(len(record["entities"]) for record in records)
log(f"Loaded {len(records):,} records with {n_entities:,} entities.")


## Stage 3 — Build the Glossary and the Masked Sentences

Tokenisation matches `common/data.py`: NFC-normalise, then split on whitespace.
Each mention is pinned to its gold span; when the gold span is unusable or its
text disagrees with `vi_term`, the first free occurrence of the term is used
instead. Mentions never overlap, so the masked sentence is unambiguous.

Every distinct mention becomes a glossary entry, and every distinct masked
sentence becomes a translation job. Placeholders are numbered in text order, so
the same sentence with the same mentions is deduplicated too.

`unlocalised` counts mentions that could not be pinned to the sentence at all;
those keep their Vietnamese term but get no English span.

In [ ]:
def normalize_text(text: str) -> str:
    """NFC-normalise and collapse whitespace; matches common/data.py."""
    return " ".join(unicodedata.normalize("NFC", text).split())


def source_span(entity):
    """The gold span, only when both indices are usable integers."""
    start = entity.get("vi_start_token_idx")
    end = entity.get("vi_end_token_idx")
    if not isinstance(start, int) or isinstance(start, bool):
        return None
    if not isinstance(end, int) or isinstance(end, bool):
        return None
    if start < 0 or end < start:
        return None
    return start, end


def free_occurrence(tokens, wanted, taken):
    """First occurrence of ``wanted`` that touches no already-taken token."""
    width = len(wanted)
    for start in range(len(tokens) - width + 1):
        if tokens[start:start + width] == wanted and not any(taken[start:start + width]):
            return start
    return None


MARKER_RE = re.compile(r"\[\[T(\d+)\]\]")


def markers_intact(source: str, result: str) -> bool:
    """Every placeholder survived, once, in the same order."""
    return MARKER_RE.findall(source) == MARKER_RE.findall(result)

prepared = []          # one entry per record: masked sentence + marker map
sentence_sources = []  # unique masked sentences to translate
term_sources = []      # unique source terms to translate (the glossary)
seen_sentences = set()
seen_terms = set()
unlocalised = 0        # mentions that could not be pinned to the sentence

for record in records:
    tokens = normalize_text(record["vi_sentence_str"]).split()
    taken = [False] * len(tokens)
    spans = []  # (start, end, entity_index)

    for entity_index, entity in enumerate(record["entities"]):
        term_key = normalize_text(entity["vi_term"])
        wanted = term_key.split()
        if term_key not in seen_terms:
            seen_terms.add(term_key)
            term_sources.append(term_key)

        span = source_span(entity)
        if span is not None:
            start, end = span
            usable = (
                end < len(tokens)
                and " ".join(tokens[start:end + 1]) == " ".join(wanted)
                and not any(taken[start:end + 1])
            )
            if not usable:
                span = None
        if span is None:
            start = free_occurrence(tokens, wanted, taken)
            if start is None:
                unlocalised += 1
                continue
            span = (start, start + len(wanted) - 1)

        start, end = span
        for position in range(start, end + 1):
            taken[position] = True
        spans.append((start, end, entity_index))

    spans.sort()
    pieces = []
    cursor = 0
    markers = {}
    for marker_index, (start, end, entity_index) in enumerate(spans):
        pieces.extend(tokens[cursor:start])
        pieces.append(f"[[T{marker_index}]]")
        markers[entity_index] = marker_index
        cursor = end + 1
    pieces.extend(tokens[cursor:])

    if not [piece for piece in pieces if not MARKER_RE.fullmatch(piece)] and len(spans) > 1:
        # Only placeholders left: nothing to translate around them, so fall back
        # to the plain sentence and lose the English spans for this record.
        unlocalised += len(spans)
        pieces = list(tokens)
        markers = {}

    masked = " ".join(pieces)
    if masked not in seen_sentences:
        seen_sentences.add(masked)
        sentence_sources.append(masked)
    prepared.append({
        "record": record,
        "masked": masked,
        "raw": normalize_text(record["vi_sentence_str"]),
        "markers": markers,
    })

raw_by_masked = {item["masked"]: item["raw"] for item in prepared}

log(
    f"Unique strings: {len(sentence_sources):,} sentences + "
    f"{len(term_sources):,} terms = {len(sentence_sources) + len(term_sources):,}."
)
if unlocalised:
    log(f"WARNING: {unlocalised:,} mention(s) could not be pinned to a source span.")

cache = {}
if cache_path.is_file():
    with cache_path.open(encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                cache[item["source"]] = item["target"]
            except (json.JSONDecodeError, KeyError, TypeError) as exc:
                raise ValueError(f"{cache_path}:{line_number}: bad cache record: {exc}") from exc
    log(f"Resumed from cache: {len(cache):,} translations already on disk.")
else:
    log("No cache yet; every string will be translated.")

pending = [text for text in term_sources + sentence_sources if not cache.get(text)]
log(f"Pending: {len(pending):,} strings.")

## Stage 4 — Load the Model

**Memory.** 27B parameters is ~54 GB in bf16 and ~15 GB in 4-bit, plus the KV
cache, so a 15 GB Colab GPU cannot hold the 4-bit build reliably. Use an
A100/L40S-class GPU, or set `MODEL_ID` to a smaller Qwen checkpoint. The
notebook logs the real footprint after loading instead of guessing.

The checkpoint is multimodal, so `AutoModelForImageTextToText` is tried first
and `AutoModelForCausalLM` is the fallback for a text-only export. Generation
itself is text-only: no images are passed.


In [ ]:
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    log(f"GPU: {torch.cuda.get_device_name(0)} "
        f"({total_b / 1024 ** 3:.1f} GiB total, {free_b / 1024 ** 3:.1f} GiB free)")
    if not LOAD_IN_4BIT and total_b / 1024 ** 3 < 60:
        log("WARNING: 27B in bf16 needs ~54 GB of weights; this GPU is too small. "
            "Set LOAD_IN_4BIT = True or pick a smaller checkpoint.")
else:
    log("WARNING: no CUDA device; a 27B model on CPU is not practical.")

log(f"Loading tokenizer and model {MODEL_ID!r} (first run downloads weights) ...")
started = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for batched decoder-only generation

load_kwargs = {
    "trust_remote_code": TRUST_REMOTE_CODE,
    "device_map": DEVICE_MAP,
    "torch_dtype": "auto",
}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

try:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
except (ImportError, ValueError) as exc:
    log(f"AutoModelForImageTextToText unavailable ({exc.__class__.__name__}); "
        "falling back to AutoModelForCausalLM.")
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

log(
    f"Model ready in {time.time() - started:.1f}s; "
    f"weights footprint {model.get_memory_footprint() / 1024 ** 3:.2f} GiB."
)


## Stage 5 — Translate the Glossary and the Sentences

Terms and masked sentences go through the same batched greedy generation; only
the system prompt differs, because a sentence carries placeholders that must
come back untouched.

A sentence result is accepted only when its placeholders come back once each, in
the same order. A rejected sentence (or an empty answer) is **not** cached, so
the next run retries it; its plain source sentence is translated as a repair so
the span can be searched instead. Every accepted translation is appended to the
cache and flushed immediately, so killing the runtime costs at most one batch.

In [ ]:
SYSTEM_PROMPT = (
    f"You are a professional biomedical translator. Translate the user's "
    f"{SOURCE_LANGUAGE} text into {TARGET_LANGUAGE}. Reply with the translation "
    "only: no notes, no quotes, no explanations. Keep gene symbols, drug names, "
    "abbreviations and numbers unchanged."
)
SENTENCE_SYSTEM_PROMPT = (
    SYSTEM_PROMPT
    + " The text contains placeholders such as [[T0]] and [[T1]] standing for "
    "biomedical terms. Copy every placeholder exactly as written, the same number "
    "of times and in the same order, and place each one where its term belongs "
    "in the translation."
)
_THINK_BLOCK = re.compile(r"<think\b[^>]*>.*?</think>", re.DOTALL | re.IGNORECASE)


def build_prompt(text: str) -> str:
    system = SENTENCE_SYSTEM_PROMPT if MARKER_RE.search(text) else SYSTEM_PROMPT
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text},
    ]
    # Qwen3-class checkpoints reason by default; the translation is the answer,
    # so thinking is disabled. Templates without that variable ignore the kwarg.
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def clean_translation(raw: str) -> str:
    text = _THINK_BLOCK.sub("", raw).strip()
    text = text.strip().strip('"').strip("'").strip()
    return " ".join(text.split())


def translate_batch(batch):
    prompts = [build_prompt(text) for text in batch]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[:, inputs["input_ids"].shape[1]:]
    return [
        clean_translation(text)
        for text in tokenizer.batch_decode(generated, skip_special_tokens=True)
    ]


def translate_all(texts, description):
    """Translate, cache, and return (empty_answers, broken_placeholders)."""
    failed = []
    broken = []
    if not texts:
        return failed, broken
    handle = cache_path.open("a", encoding="utf-8")
    try:
        with tqdm(total=len(texts), desc=description, unit="string") as bar:
            for start in range(0, len(texts), BATCH_SIZE):
                batch = texts[start:start + BATCH_SIZE]
                try:
                    results = translate_batch(batch)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    log(f"OOM on a batch of {len(batch)}; retrying one string at a time.")
                    results = []
                    for text in batch:
                        try:
                            results.extend(translate_batch([text]))
                        except torch.cuda.OutOfMemoryError:
                            torch.cuda.empty_cache()
                            results.append("")
                for text, result in zip(batch, results):
                    if not result:
                        failed.append(text)
                        continue
                    if MARKER_RE.search(text) and not markers_intact(text, result):
                        broken.append(text)
                        continue
                    cache[text] = result
                    handle.write(json.dumps({"source": text, "target": result}, ensure_ascii=False) + "\n")
                    handle.flush()
                bar.update(len(batch))
    finally:
        handle.close()
    return failed, broken


started = time.time()
failures, broken_markers = translate_all(pending, "Translating")
elapsed = time.time() - started
log(
    f"Translated {len(pending) - len(failures) - len(broken_markers):,} of "
    f"{len(pending):,} pending strings in {elapsed:.1f}s "
    f"({len(pending) / max(elapsed, 1e-6):.2f} strings/s)."
)

repair_texts = []
for text in broken_markers:
    raw = raw_by_masked.get(text)
    if raw and raw != text and not cache.get(raw) and raw not in repair_texts:
        repair_texts.append(raw)
if broken_markers:
    log(
        f"WARNING: {len(broken_markers):,} sentence(s) dropped or moved a placeholder "
        f"and are not cached; {len(repair_texts):,} plain sentence(s) will be "
        "translated instead so their spans can be searched."
    )
if repair_texts:
    repair_failures, _ = translate_all(repair_texts, "Repairing")
    failures.extend(repair_failures)

if failures:
    log(f"WARNING: {len(failures):,} empty answer(s); first: {failures[0]!r}. "
        "They are not cached and will be retried on the next run.")

## Stage 6 — Assemble the Translated Dataset

Every original field is copied unchanged (`cui`, `icd10`, `label`, and both
Vietnamese strings and indices included). `en_term` and `en_sentence_str` are
added, and the English span is read off the substitution: the placeholder sits
where the term belongs, so its token position is the term position. No search,
no guessing, and repeated mentions keep their own position.

A record whose placeholders did not survive falls back to its plain translation
and a first-occurrence search; a mention that still cannot be located keeps
`null` indices, is counted, and is reported with its record and entity.

In [ ]:
def isolate_markers(text: str) -> str:
    """Space out every placeholder so it tokenises on its own."""
    return re.sub(r"\s*(\[\[T\d+\]\])\s*", r" \1 ", text).strip()


def substitute_markers(text: str, term_by_index):
    """Replace each placeholder with its English term, recording the span.

    Returns ``(sentence, {marker index: (start, end)})``. The span is exact
    because the term is inserted here, not searched for afterwards.
    """
    tokens = []
    spans = {}
    for piece in isolate_markers(text).split():
        match = MARKER_RE.fullmatch(piece)
        if match is None:
            tokens.append(piece)
            continue
        marker_index = int(match.group(1))
        start = len(tokens)
        tokens.extend(term_by_index[marker_index].split())
        spans[marker_index] = (start, len(tokens) - 1)
    return " ".join(tokens), spans


def find_span(tokens, term):
    """First-occurrence fallback for records without placeholders."""
    wanted = term.split()
    width = len(wanted)
    for start in range(len(tokens) - width + 1):
        if tokens[start:start + width] == wanted:
            return start, start + width - 1
    return None, None


translated_records = []
marker_spans = 0
searched_spans = 0
unmatched = []
missing = 0

for record_index, item in enumerate(prepared):
    record = item["record"]
    markers = item["markers"]
    terms = [cache.get(normalize_text(entity["vi_term"])) for entity in record["entities"]]

    masked_translation = cache.get(item["masked"]) if markers else None
    use_markers = bool(
        markers
        and masked_translation
        and markers_intact(item["masked"], masked_translation)
        and all(terms[index] for index in markers.values())
    )
    if use_markers:
        term_by_index = {index: terms[entity_index] for entity_index, index in markers.items()}
        sentence, span_by_marker = substitute_markers(masked_translation, term_by_index)
    else:
        sentence = cache.get(item["raw"])
        span_by_marker = {}
    if not sentence:
        missing += 1
        continue

    tokens = sentence.split()
    entities = []
    for entity_index, entity in enumerate(record["entities"]):
        term = terms[entity_index]
        if not term:
            missing += 1
            continue
        if use_markers and entity_index in markers:
            start, end = span_by_marker[markers[entity_index]]
            marker_spans += 1
        else:
            start, end = find_span(tokens, term)
            searched_spans += 1
            if start is None:
                unmatched.append((record_index, entity_index, term))
        entities.append({
            **entity,
            "en_term": term,
            "en_start_token_idx": start,
            "en_end_token_idx": end,
        })
    translated_records.append({
        **record,
        "en_sentence_str": sentence,
        "entities": entities,
    })

if missing:
    raise RuntimeError(
        f"{missing:,} string(s) still have no translation. Re-run stage 5 until "
        "it reports zero pending strings, then assemble again."
    )

with output_path.open("w", encoding="utf-8") as handle:
    for record in translated_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

log(
    f"Wrote {len(translated_records):,} records "
    f"({output_path.stat().st_size / 1024 ** 2:.2f} MB) to {output_path}."
)
log(
    f"Spans: {marker_spans:,} from placeholders, {searched_spans:,} searched, "
    f"{len(unmatched):,} null."
)
if unmatched:
    log(
        f"WARNING: {len(unmatched):,} entity span(s) could not be located and are "
        f"null; first: record {unmatched[0][0]} entity {unmatched[0][1]} "
        f"term {unmatched[0][2]!r}."
    )
else:
    log("All entity spans were located.")

## Stage 7 — Verify and Release Memory

Reload the written file and check what a consumer would rely on: every record
has an English sentence, every entity has an English term, and every English
span is in range and covers exactly that term. Then drop the model, the
tokenizer and the records and empty the CUDA cache.

In [ ]:
log(f"Reloading {output_path} ...")
with output_path.open(encoding="utf-8") as handle:
    reloaded = [json.loads(line) for line in handle if line.strip()]
assert len(reloaded) == len(translated_records), (len(reloaded), len(translated_records))

bad_spans = []
for record_index, record in enumerate(reloaded):
    assert record.get("en_sentence_str"), record_index
    tokens = record["en_sentence_str"].split()
    for entity_index, entity in enumerate(record["entities"]):
        assert entity.get("en_term"), (record_index, entity_index)
        start = entity["en_start_token_idx"]
        end = entity["en_end_token_idx"]
        if start is None or end is None:
            bad_spans.append((record_index, entity_index, "null"))
            continue
        if not (0 <= start <= end < len(tokens)):
            bad_spans.append((record_index, entity_index, "out of range"))
            continue
        if " ".join(tokens[start:end + 1]) != entity["en_term"]:
            bad_spans.append((record_index, entity_index, "term mismatch"))

log(f"Verified: {len(reloaded):,} translated records, cache {len(cache):,} strings.")
if bad_spans:
    log(
        f"WARNING: {len(bad_spans):,} English span(s) are not usable; first: "
        f"record {bad_spans[0][0]} entity {bad_spans[0][1]} ({bad_spans[0][2]})."
    )
else:
    log("Every English span covers exactly its term.")

for name in ("model", "tokenizer", "records", "prepared", "translated_records", "reloaded", "inputs", "output"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

log("Translation complete; memory released.")